In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ជំហានទី 1 ៖ កំណត់ម៉ូដែល Pydantic សម្រាប់លទ្ធផលដែលមានរចនាសម្ព័ន្ធ

ម៉ូដែលទាំងនេះបានកំណត់ **ស្កីមា** ដែលភ្នាក់ងារនឹងបង្រួមតបតាម។ ការប្រើ `response_format` ជាមួយ Pydantic ធានាថា៖
- ✅ ការដកស្រង់ទិន្នន័យដែលមានប្រភេទសុវត្ថិភាព
- ✅ ការផ្ទៀងផ្ទាត់ដោយស្វ័យប្រវត្តិ
- ✅ គ្មានកំហុសបំភ្លឺពីចម្លើយអត្ថបទសេរី
- ✅ ការបែងចែកលំហូរងាយៗដោយផ្អែកលើវាលបញ្ជូល


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ជំហ៊ាន 2: បង្កើតឧបករណ៍កក់សណ្ឋាគារ

ឧបករណ៍នេះជាអ្វីដែល **availability_agent** នឹងហៅដើម្បីពិនិត្យមើលថាតើមានបន្ទប់នៅសល់ទេ។ យើងប្រើកម្មវិធី `@ai_function` ដើម្បី:
- បម្លែងមុខងារ Python មួយទៅជាឧបករណ៍ដែលអាចហៅបានដោយ AI
- បង្កើតសchema JSON ស្វ័យប្រវត្តិសម្រាប់ LLM
- ដោះស្រាយការត្រួតពិនិត្យប៉ារ៉ាម៉ែត្រ
- អនុញ្ញាតឱ្យមានការហៅដោយស្វ័យប្រវត្តិដោយតំណាង

សម្រាប់ការបង្ហាញនេះ:
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → មានបន្ទប់ ✅
- **ទីក្រុងផ្សេងទៀតទាំងអស់** → មិនមានបន្ទប់ ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ជំហ៊ានទី 3: កំណត់មុខងារជ្រើសរើសលក្ខខណ្ឌសម្រាប់ការបញ្ជូនផ្លូវ

មុខងារទាំងនេះពិនិត្យមើលការឆ្លើយតបរបស់ភ្នាក់ងារ ហើយកំណត់ថាតើគួរតែទៅផ្លូវណា ក្នុងដំណើរការងារ។

**រចនាប័ទ្មសំខាន់៖**
១. ផ្ទៀងផ្ទាត់មើលថា សារ​ជាប្រភេទ `AgentExecutorResponse` នោះឬទេ
២. ព្រមានលទ្ធផលដែលបានរៀបចំ (ម៉ូដែល Pydantic)
៣. ត្រឡប់តម្លៃជា `True` ឬ `False` ដើម្បីគ្រប់គ្រងការបញ្ជូនផ្លូវ

ដំណើរការងារនឹងវាយតម្លៃលក្ខខណ្ឌទាំងនេះនៅលើ **គេភ្ជាប់** ដើម្បីសម្រេចចិត្តថាតើគួរតែរំដោះកម្មវិធីណាដើម្បីប្រើបន្ទាប់។


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## ជាន់ទី 4: បង្កើតកម្មវត្ថុដំណើរការបង្ហាញផ្ទាល់ខ្លួន

Executors គឺជាប៉ារ៉ាកម្ម workflow ដែលបំពេញការបម្លែងឬប្លែកភាពផ្សេងៗ។ យើងប្រើ `@executor` ដើម្បីបង្កើតកម្មវត្ថុដំណើរការផ្ទាល់ខ្លួនដែលបង្ហាញលទ្ធផលចុងក្រោយ។

**មូលដ្ឋានគន្លងចម្បង៖**
- `@executor(id="...")` - ចុះបញ្ជីមុខងារជា executor តាម workflow
- `WorkflowContext[Never, str]` - ការប្រុងប្រយ័ត្នប្រភេទសម្រាប់បញ្ចូល/ចេញ
- `ctx.yield_output(...)` - ផ្ដល់លទ្ធផលចុងក្រោយនៃ workflow


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## ជំហានទី 5៖ បញ្ចូលអថេរបរិស្ថាន

កំណត់រចនាសម្ព័ន្ធកម្មវិធីអតិថិជន LLM។ ឧទាហរណ៍នេះធ្វើការជាមួយ:
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — ការផ្តល់អាទិភាព)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ជំហានទី 6: បង្កើតភ្នាក់ងារតាម AI ជាមួយនឹងលទ្ធផលដែលរៀបរយ

យើងបង្កើត **ភ្នាក់ងារពិសេសបីនាក់** ផ្តួចផ្តើមក្នុង `AgentExecutor`៖

1. **availability_agent** - ពិនិត្យមើលភាពទំនេរនៃសណ្ឋាគារដោយប្រើឧបករណ៍
2. **alternative_agent** - ការផ្ដល់អាទិភាពទីក្រុងជម្រើសបន្ថែម (ពេលគ្មានបន្ទប់)
3. **booking_agent** - ជម្រុញការកក់បន្ទប់ (ពេលមានបន្ទប់ទំនេរ)

**លក្ខណៈសំខាន់ៗ:**
- `tools=[hotel_booking]` - ផ្តល់ឧបករណ៍ទៅភ្នាក់ងារ
- `response_format=PydanticModel` - បង្ខំលទ្ធផល JSON តាមរចនាសម្ព័ន្ធ
- `AgentExecutor(..., id="...")` - យកភ្នាក់ងាររាក់ទាក់សម្រាប់ប្រើក្នុងលំនាំកម្មវិធី


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ជំហ៊ាន 7៖ សាងសង់សកម្មភាពជាមួយខ្សែភាពជាប់លក្ខខណ្ឌ

ឥឡូវនេះយើងប្រើ `WorkflowBuilder` ដើម្បីសាងសង់ក្រាហ្វជាមួយបណ្តោយដែលអាស្រ័យលើលក្ខខណ្ឌ៖

**រចនាសម្ព័ន្ធសកម្មភាព៖**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**វិធីសាស្រ្តសំខាន់ៗ៖**
- `.set_start_executor(...)` - កំណត់ចំណុចចូល
- `.add_edge(from, to, condition=...)` - បន្ថែមខ្សែភាពជាប់លក្ខខណ្ឌ
- `.build()` - បញ្ចប់សកម្មភាព


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## ជំហាន 8៖ បើករត់ករណីសាកល្បង 1 - ទីក្រុងដោយគ្មានការចូលប្រើ (ប៉ារីស)

យើងមកសាកល្បងផ្លូវ **គ្មានការចូលប្រើ** ដោយស្នើសុំសណ្ឋាគារនៅប៉ារីស (ដែលគ្មានបន្ទប់នៅក្នុងការសំយោគរបស់យើង)។


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## ជំហានទី 9៖ ប្រតិបត្តិរឿងសាកល្បង 2 - ទីក្រុង ជាមួយ មានស្រាប់ (Stockholm)

ឥឡូវនេះ យើងនឹងសាកល្បងផ្លូវ **availability** ដោយស្នើសុំសណ្ឋាគារនៅ Stockholm (ដែលមានបន្ទប់ក្នុងការសម្របសម្រួលរបស់យើង)។


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ចំណុចសំខាន់ៗ និងជំហានបន្ទាប់

### ✅ អ្វីដែលអ្នកបានរៀន៖

1. **គំរូ WorkflowBuilder**
   - ប្រើ `.set_start_executor()` ដើម្បីកំណត់ចំណុចចាប់ផ្តើម
   - ប្រើ `.add_edge(from, to, condition=...)` សម្រាប់ការបញ្ជូនតាមលក្ខខណ្ឌ
   - ហៅ `.build()` ដើម្បីបញ្ចប់ workflow

2. **ការបញ្ជូនតាមលក្ខខណ្ឌ**
   - មុខងារ​លក្ខខណ្ឌពិនិត្យ `AgentExecutorResponse`
   - បញ្ជូរផលិតផលដែលមានរចនា ដើម្បីធ្វើការសម្រេចបញ្ជូន
   - ត្រឡប់ `True` ដើម្បីបើកអេដចុង, `False` ដើម្បីរំលងវា

3. **ការរួមបញ្ចូលឧបករណ៍**
   - ប្រើ `@ai_function` ដើម្បីបម្លែងមុខងារ Python ទៅជាឧបករណ៍ AI
   - អ្នកប្រើហៅឧបករណ៍ដោយស្វ័យប្រវត្តិពេលចាំបាច់
   - ឧបករណ៍​ត្រឡប់ JSON ដែលអ្នកប្រើអាចបញ្ជូរ

4. **ផលិតផលដែលមានរចនា**
   - ប្រើម៉ូទូច Pydantic សម្រាប់បញ្ចេញទិន្នន័យដែលគ្មានបញ្ហាប្រភេទ
   - កំណត់ `response_format=MyModel` នៅពេលបង្កើតអ្នកប្រើ
   - បញ្ជូរឆ្លើយតបជាមួយ `Model.model_validate_json()`

5. **អ្នកបើកប្រេតកម្មផ្ទាល់ខ្លួន**
   - ប្រើ `@executor(id="...")` ដើម្បីបង្កើតផ្នែកចូលរួម workflow
   - អ្នកបើកប្រេតអាចបម្លែងទិន្នន័យ ឬអនុវត្តប៉ះពាល់មួយចំនួន
   - ប្រើ `ctx.yield_output()` ដើម្បីផលិតលទ្ធផល workflow

### 🚀 ការអនុវត្តក្នុងពិភពលោកពិត៖

- **កក់ការធ្វើដំណើរ**៖ ពិនិត្យភាពអាចប្រើបាន, ស្នើជម្រើសជំនួស, ប្រៀបធៀបជម្រើស
- **សេវាកម្មអតិថិជន**៖ ចែកផ្លូវតាមប្រភេទបញ្ហា, អារម្មណ៍, លំដាប់អាទិភាព
- **អ៊ី-គមមែប**៖ ពិនិត្យស្តុកសម្ភារ, ស្នើជម្រើសជំនួស, ដំណើរការបញ្ជាទិញ
- **ត្រួតពិនិត្យមាតិកា**៖ ចែកផ្លូវតាមពិន្ទុអំពើអាក្រក់, សញ្ញាអ្នកប្រើ
- **ដំណើរការយល់ព្រម**៖ ចែកផ្លូវតាមចំនួនទឹកប្រាក់, តួនាទីអ្នកប្រើ, កម្រិតហានិភ័យ
- **ដំណើរការពហុជំហាន**៖ ចែកផ្លូវតាមគុណភាពទិន្នន័យ, ការបំពេញ

### 📚 ជំហានបន្ទាប់៖

- បន្ថែមលក្ខខណ្ឌស្មុគស្មាញ (លក្ខខណ្ឌច្រើន)
- អនុវត្តរង្វិលជាមួយការគ្រប់គ្រងស្ថានភាព workflow
- បន្ថែម sub-workflows សម្រាប់ផ្នែកដែលអាចប្រើឡើងវិញ
- ភ្ជាប់ជាមួយ API ពិត (កក់សណ្ឋាគារ, ប្រព័ន្ធស្តុក)
- បន្ថែមការដោះស្រាយកំហុស និងផ្លូវព្រាង
- មើលរូបភាព workflows ជាមួយឧបករណ៍វិចិត្រសាលក្នុងស្រាប់


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
